# ASR

Это демо ноутбук проекта ASR. Он призван помочь объяснить как использовать данный проект.

### Установка зависимостей

Рекомендуется использовать этот репозиторий в conda-окружении или в venv. Однако для понятности кода в этом ноутбуке эта часть опускается. Рекомендуется запускать это демо в `Google Colab`. Если он запускается из уже скачанного проекта, выполните команду

```bash
%cd ..
```

Давайте скачаем репозиторий проекта (если он еще не скачан).

In [ ]:
repo_url = f"https://github.com/MrKustic/asr.git"
!git clone $repo_url
%cd asr

fatal: destination path 'asr' already exists and is not an empty directory.
/content/asr


Установим все необходимые зависимости (в colab возможно придется перезапустить сеанс).

In [ ]:
%pip install -r requirements.txt

  Using cached pyctcdecode-0.6.0-py2.py3-none-any.whl


### Скачивание датасета

Все необходимые чекпоинты моделей и примеры расположены в [Hagging Face репозитории](https://huggingface.co/MrKustic/asr). Давайте скачаем чекпоинт модели, токенизатор и папку с небольшим примером. Чтобы увеличить скорость скачивания рекомендуется залогинится на HF, раскомментировав и выполнив код в следующей ячейке. Если вы запускаете ноутбук не в Colab, можно залогинится интерактивно, выполнив такой код

```python
from huggingface_hub import login

login()
```

In [ ]:
# from google.colab import userdata
# from huggingface_hub import login

# hf_token = userdata.get('hf_token')
# login(hf_token)

In [ ]:
# путь до корня проекта
ROOT_PATH = %pwd

%cd scripts/

# Скачиваем чекпоинт модели
!python download_hf_folder.py \
    --username "MrKustic" \
    --project "asr" \
    --folder "model" \
    --local-dir "{ROOT_PATH}/data/demo"

# Скачиваем токенизатор
!python download_hf_folder.py \
    --username "MrKustic" \
    --project "asr" \
    --folder "tokenizers" \
    --local-dir "{ROOT_PATH}/data/demo"

# Скачиваем папку с ручным датасетом
!python download_hf_folder.py \
    --username "MrKustic" \
    --project "asr" \
    --folder "example" \
    --local-dir "{ROOT_PATH}/data/demo"

### Демонстрационный пример

In [ ]:
%cd "{ROOT_PATH}/data/demo/example"
!ls

%cd "audio"
!ls

%cd "../transcriptions"
!ls

Как видно, мы скачали демонстрационный датасет из двух аудио и их транскрипций. Аудио `clear.flac` взято из части `dev-clean` датасета `librispeech`, тогда как `other.wav` записано нами. Покажем последний подробнее.

In [ ]:
from IPython.display import Audio
from pathlib import Path

demo_dir = Path(ROOT_PATH) / "data" / "demo"

with (demo_dir / "example" / "transcriptions" / "other.txt").open("r") as f:
    print(f.readline().strip())

Audio(demo_dir / "example" / "audio" / "other.wav")

Давайте запустим на этом датасете нашу модель. Для этого используем скрипт inference.py. Далее для возможности оставлять комментарии будем передавать аргументы списком.

In [ ]:
# Удобный хелпер, позволяет запускать subprocess и параллельно выводить stdout
def run_and_print(args):
    command = [args[0], "-u", *args[1:]]

    with subprocess.Popen(command, cwd=str(ROOT_PATH), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)

        returncode = process.wait()
        if returncode != 0:
            raise subprocess.CalledProcessError(returncode, command)

In [ ]:
import subprocess

args = [
    "python",
    "inference.py",

    # входные данные
    "datasets=custom_dataset",
    f"datasets.inference.audio_dir={demo_dir}/example/audio",
    f"+datasets.inference.transcription_dir={demo_dir}/example/transcriptions",

    # модель и токенизатор
    f"inferencer.from_pretrained={demo_dir}/model/model_best.pth",
    f"tokenizer.model_path={demo_dir}/tokenizers/tokenizer-960.model",

    # каталог для предсказаний
    f"+inferencer.save_path={demo_dir}/example/predictions",
]

run_and_print(args)

Чтобы улучшить полученные метрики CER и WER можно добавить beam search и внешнюю языковую модель для его корректировки. Предварительно скачаем эту модель.

In [ ]:
%cd "{ROOT_PATH}/scripts"
!python download_hf_folder.py \
    --username "MrKustic" \
    --project "asr" \
    --folder "lm" \
    --local-dir "{demo_dir}"

Запустим наш скрипт с beam search и language model.

In [ ]:
args = [
    "python",
    "inference.py",

    # входные данные
    "datasets=custom_dataset",
    f"datasets.inference.audio_dir={demo_dir}/example/audio",
    f"+datasets.inference.transcription_dir={demo_dir}/example/transcriptions",

    # модель и токенизатор
    f"inferencer.from_pretrained={demo_dir}/model/model_best.pth",
    f"tokenizer.model_path={demo_dir}/tokenizers/tokenizer-960.model",

    # каталог для предсказаний
    f"+inferencer.save_path={demo_dir}/example/predictions",

    # параметры beam search и lm
    f"ctc_decoder=beam_lm",
    f"ctc_decoder.kenlm_model_path={demo_dir}/lm/3-gram.lower.arpa"
]

run_and_print(args)

Посмотрим на предсказания. Также для правильного сравнения будем нормализовывать оба текста в паре.

In [ ]:
from asr.src.utils.text import normalize_text

for name in ["clear", "other"]:
    print(name)
    with (Path(demo_dir) / "example" / "transcriptions" / f"{name}.txt").open("r") as f:
        transcription = normalize_text(f.readline())
        print(f"Transcription: {transcription}")
    with (Path(demo_dir) / "example" / "predictions" / "inference" / f"{name}.txt").open("r") as f:
        prediction = normalize_text(f.readline())
        print(f"Prediction:    {prediction}")
    print()

### Собственный датасет

Также предагается запустить модель на кастомном датасете. Для этого нужно указать ссылку на ваш датасет. В корне должна быть папка `audio` с аудио формата `.wav`, `.flac` или `.mp3`. Также (опционально) в корне может быть папка `transcriptions`, которая содержит все транскрипции аудио (из папки `audio`) в формате `.txt`. При этом файлы аудио и его транскрипции должны иметь одинаковые назавния (не считая расширения).

In [ ]:
import gdown
import os

custom_dataset_url = "https://drive.google.com/drive/folders/15qMJDVHXtdgZRvyfCKJbx5W5DCKHeJL8?usp=sharing" # укажите ссылку на ваш датасет
custom_dataset_dir = f"{demo_dir}/custom_dataset"

gdown.download_folder(url=custom_dataset_url, output=custom_dataset_dir)
folder_names = os.listdir(custom_dataset_dir)

local_dir = Path(custom_dataset_dir)
if not (local_dir / "audio").is_dir():
    folders_with_audio = [
        path for path in local_dir.iterdir()
        if path.is_dir() and (path / "audio").is_dir()
    ]
    if len(folders_with_audio) != 1:
        raise ValueError("Не удалось определить папку датасета с audio/")
    local_dir = folders_with_audio[0]

In [ ]:
PREDICTIONS_DIR = f"{local_dir}/predictions" # укажите директорию для сохранения предсказаний

args = [
    "python",
    "inference.py",

    # входные данные
    "datasets=custom_dataset",
    f"datasets.inference.audio_dir={local_dir}/audio",

    # модель и токенизатор
    f"inferencer.from_pretrained={demo_dir}/model/model_best.pth",
    f"tokenizer.model_path={demo_dir}/tokenizers/tokenizer-960.model",

    # каталог для предсказаний
    f"+inferencer.save_path={PREDICTIONS_DIR}",

    # параметры beam search и lm
    f"ctc_decoder=beam_lm",
    f"ctc_decoder.kenlm_model_path={demo_dir}/lm/3-gram.lower.arpa",

    # не считаем метрики при этом запуске
    f"metrics.inference=[]"
]

run_and_print(args)

Тогда метрики можно вычислить так:

In [ ]:
TRANSCRIPTIONS_DIR = f"{local_dir}/transcriptions" # укажите директорию с транскрипциями

if Path(TRANSCRIPTIONS_DIR).exists():
    args = [
        "python",
        "calc_metrics.py",

        # входные данные
        "datasets=custom_dataset",
        f"datasets.inference.audio_dir={local_dir}/audio",
        f"datasets.inference.transcription_dir={TRANSCRIPTIONS_DIR}",

        # только токенизатор
        f"tokenizer.model_path={demo_dir}/tokenizers/tokenizer-960.model",

        # каталог для предсказаний
        f"+inferencer.save_path={PREDICTIONS_DIR}",
    ]

    run_and_print(args)
else:
    print("Directory with transcriptions does not exist.")